In [ ]:
# 경고 메시지 무시
import warnings
warnings.filterwarnings(action='ignore') 

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import csv
import folium
import datetime
import seaborn as sns
import scipy as sp
import statsmodels.formula.api as smf
import networkx as nx
import missingno as msno
import os
import sys
import urllib.request
import time
import json
import plotly.express as px
import re

from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes
from folium.plugins import HeatMap 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler 
from dateutil.relativedelta import relativedelta
from sklearn.cluster import KMeans    ##  K-means 임포트
from sklearn.metrics import silhouette_score
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D
from operator import itemgetter
from PIL import Image
from collections import Counter
from wordcloud import WordCloud
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

plt.rc('font',family='D2CodingLigature Nerd Font')
# plt.rc('font', family='malgun gothic')
# plt.rcParams['axes.unicode_minus']=False  # '- 표시

## 서울시 외국인 인구수 데이터

In [ ]:
geo_seoul = json.load(open('../../data/EMD_Seoul.geojson', encoding = 'UTF-8'))

In [ ]:
# 서울시 동별 외국인 인구 데이터 준비하기
foreigner = pd.read_csv('../../data/Foreigner_EMD_Seoul.csv')
foreigner.head()

In [ ]:
foreigner.info()

In [ ]:
# code항목을 문자 타입으로 바꾸기
foreigner['code'] = foreigner['code'].astype(str)
foreigner.info()

In [ ]:
map_seoul1 = folium.Map(location = [37.56, 127], zoom_start = 12, tiles = 'cartodbpositron')

folium.Choropleth(geo_data = geo_seoul,
                  data = foreigner,
                  columns = ('code', 'pop'),
                  key_on = 'feature.properties.ADM_DR_CD',
                  fill_color = 'Blues',
                  bins = list(foreigner['pop'].quantile([0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])),
                  nan_fill_color = 'White',
                  fill_opacity = 1,
                  line_opacity = 0.5
                  ) \
      .add_to(map_seoul1)

map_seoul1

In [ ]:
# 구 경계선 추가하기
# SIG_Seoul.geojson: 서울시의 구 경계 좌표를 담고 있음
geo_seoul_sig = json.load(open('../../data/SIG_Seoul.geojson', encoding = 'UTF-8'))

folium.Choropleth(geo_data = geo_seoul_sig,
                  fill_opacity = 0,
                  line_weight = 2,
                  line_color = 'yellow') \
      .add_to(map_seoul1)

map_seoul1

In [ ]:
# GeoJSON
for feature in geo_seoul['features']:
    sig_cd = feature['properties']['ADM_DR_CD']
    pop_data = foreigner[foreigner['code']==sig_cd]

    if not pop_data.empty:
        pop_value = pop_data['pop'].values[0]
        if pd.notna(pop_value):
            feature['properties']['pop'] = int(pop_value)

In [ ]:
# 구간 정하기 (8단계)
b = list(foreigner['pop'].quantile([0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]))

# 지도 생성
map_seoul2 = folium.Map(location=[37.56, 127], zoom_start=12, tiles='cartodbpositron')

folium.Choropleth(
    geo_data=geo_seoul,
    data=foreigner,
    columns=('code', 'pop'),
    key_on='feature.properties.ADM_DR_CD',
    fill_color='Blues',
    nan_fill_color='White',
    fill_opacity=1,
    line_opacity=0.5,
    line_weight = 0.1,
    bins=b
).add_to(map_seoul2)

# 툴팁에 외국인 인구 수 추가
folium.GeoJson(
    geo_seoul,
    name="서울시 동별 외국인 인구",
    tooltip=folium.GeoJsonTooltip(
        fields=["ADM_DR_NM", "pop"],  # 동 이름과 외국인 인구 필드 추가
        aliases=["동 이름: ", "외국인 인구: "],  # 툴팁에 보일 레이블
        localize=True,
        labels=True
    )
).add_to(map_seoul2)

# 지도 출력
map_seoul2